In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# 1. Leemos los sectores únicos reales que vienen desde Silver
df_sectores_sunat = spark.table("silver.sunat_paises").select("sector").distinct()
df_sectores_comtrade = spark.table("silver.comtrade").select("sector").distinct()

# 2. Unimos ambas fuentes y quitamos duplicados
df_sectores_unicos = df_sectores_sunat.union(df_sectores_comtrade).distinct()

# 3. Generamos la clave sustituta (id_sector)
ventana = Window.orderBy("sector")
dim_producto_sector = df_sectores_unicos.withColumn("id_sector", F.row_number().over(ventana))

# Ordenamos las columnas (ID primero)
dim_producto_sector = dim_producto_sector.select("id_sector", "sector")

# Guardamos en Gold
dim_producto_sector.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_producto_sector")

print(f"Dimensión Sector generada con {dim_producto_sector.count()} sectores.")
display(dim_producto_sector)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Dimensión Sector generada con 2 sectores.


id_sector,sector
1,Café
2,Total General
